# Bài 10: Mạng học như thế nào
**Mở hộp đen AI · Tập 2 · Phòng thí nghiệm Colab**

Bốn chặng: **Chạy → Đọc → Sửa → Tự làm**. Sổ bán trà sữa là số liệu tự soạn của Bài 4, Tập 1.
Ebook Bên trong cỗ máy Trí tuệ nhân tạo - Tập 2. Phát hành tại https://montinhoc.vn

## Chặng 1: Chạy
### Ô 1. Gặp lại quán trà sữa, và một thước đo độ sai

In [ ]:
import numpy as np

nhiet_do = np.array([24, 25, 26, 26, 27, 28, 28, 29, 30, 30,
                     31, 31, 32, 32, 33, 33, 34, 35, 35, 36])
so_ly = np.array([50, 58, 60, 65, 64, 72, 69, 77, 78, 83,
                  82, 88, 87, 92, 95, 91, 100, 101, 106, 107])
x = nhiet_do - 30                  # dời gốc về 30 độ để máy dễ học (xem mục 5.5)

def mat_mat(a, b):
    # trung bình bình phương sai số (MSE)
    return np.mean((so_ly - (a * x + b)) ** 2)

print("Mất mát khi a = 0, b = 0  :", round(mat_mat(0, 0), 1))
print("Mất mát khi a = 4, b = 80 :", round(mat_mat(4, 80), 1))

### Ô 2. Đo độ dốc bằng cách nhích thử

In [ ]:
def do_doc(a, b, h=1e-4):
    da = (mat_mat(a + h, b) - mat_mat(a - h, b)) / (2 * h)
    db = (mat_mat(a, b + h) - mat_mat(a, b - h)) / (2 * h)
    return da, db

da, db = do_doc(0, 0)
print(f"Tại a = 0, b = 0: dốc theo a = {da:.1f}, dốc theo b = {db:.1f}")

### Ô 3. Đi xuống dốc (gradient descent)

In [ ]:
def xuong_doc(toc_do, so_buoc=200, in_moi=None):
    a, b = 0.0, 0.0
    lich_su = []
    for buoc in range(so_buoc):
        da, db = do_doc(a, b)
        a = a - toc_do * da          # bước ngược chiều dốc
        b = b - toc_do * db
        lich_su.append(mat_mat(a, b))
        if lich_su[-1] > 1e12:       # mất mát bùng nổ: dừng lại
            break
        if in_moi and (buoc + 1) % in_moi == 0:
            print(f"bước {buoc + 1:3}: a = {a:6.2f}, b = {b:6.2f},"
                  f" mất mát = {lich_su[-1]:8.2f}")
    return a, b, lich_su

a, b, lich_su = xuong_doc(toc_do=0.05, in_moi=40)
print(f"Đổi về nhiệt độ thật: số ly = {a:.2f} x nhiệt độ + ({b - 30 * a:.2f})")

## Chặng 2: Đọc
1. Ở ô 2, dốc theo a và theo b mang dấu âm. Điều đó nói gì về hướng phải đi?
2. Ở ô 3, dòng nào là một **bước học**? Máy có giải phương trình nào không?
3. So kết quả cuối ô 3 với đường thẳng của Bài 4 (4,65 và −59,53).

## Chặng 3: Sửa
### Ô 4. Tốc độ học: quá nhỏ, vừa phải, quá lớn

In [ ]:
for toc_do in [0.001, 0.05, 0.6]:
    a, b, ls = xuong_doc(toc_do, so_buoc=50)
    print(f"tốc độ {toc_do:<5}: sau {len(ls)} bước, mất mát = {ls[-1]:.3g}")

### Ô 5. Nhiều nơ-ron giải được bài XOR mà một nơ-ron chịu thua (Bài 9)

In [ ]:
from sklearn.neural_network import MLPClassifier

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])
mang = MLPClassifier(hidden_layer_sizes=(4,), activation="tanh", solver="lbfgs",
                     max_iter=1000, random_state=1).fit(X, y_xor)
print("Đoán:", mang.predict(X), "| Đáp án:", y_xor)

### Ô 6. Nhận đủ 10 chữ số: một lớp và nhiều lớp

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
import warnings
warnings.filterwarnings("ignore")   # ẩn cảnh báo "chưa hội tụ" (xem mục 5.6)

so = load_digits()
X_hoc, X_thi, y_hoc, y_thi = train_test_split(so.data / 16, so.target,
                                              test_size=0.3, random_state=0)
p = Perceptron(random_state=0).fit(X_hoc, y_hoc)
mlp = MLPClassifier(hidden_layer_sizes=(32,), max_iter=300, random_state=0)
mlp.fit(X_hoc, y_hoc)
print("Perceptron một lớp      :", round(p.score(X_thi, y_thi), 3))
print("Mạng 1 lớp ẩn, 32 nơ-ron:", round(mlp.score(X_thi, y_thi), 3))
so_tham_so = sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
print("Số trọng số của mạng    :", so_tham_so)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(mlp.loss_curve_)
plt.xlabel("Vòng học"); plt.ylabel("Mất mát")
plt.title("Mạng học dần qua từng vòng"); plt.show()

## Chặng 4: Tự làm
1. Ở ô 6, thử `hidden_layer_sizes` bằng (4,), (16,), (64,), (32, 32). Ghi điểm vào bảng. Thêm nơ-ron có luôn tốt hơn không?
2. Ở ô 4, tìm tốc độ học lớn nhất mà máy vẫn đi xuống được.
3. Ở ô 5, đổi `hidden_layer_sizes=(4,)` thành `(2,)`, rồi thử `random_state` từ 0 đến 7. Lần nào mạng giải sai? Vì sao kết quả lại phụ thuộc vào điểm xuất phát?

**Nhật ký dùng AI:** Em có nhờ AI ở bước nào không? Nó sai ở đâu? Em sửa thế nào?

*(ghi vào đây)*